# Part 4 — GenAI: Structured Follow-up Note Analysis

Tasks 4.1–4.4. This notebook requires a free Gemini API key entered at runtime. **Never paste the key into a code cell, notebook text, or GitHub.** Clear notebook outputs before committing. The first 10 call-centre notes are analysed; blank note N004 is skipped without an API call.

In [ ]:
!pip install -q google-genai pydantic

import json
import time
import pandas as pd
from pathlib import Path
from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types

import os
import subprocess
REPO_URL = 'https://github.com/Subham9339063436/CareLoop-Clinics-No-Show-Guard.git'
REPO_DIR = Path('/content/CareLoop-Clinics-No-Show-Guard')
if not (Path.cwd() / 'data' / 'followup_notes.csv').exists():
    if not (REPO_DIR / 'data' / 'followup_notes.csv').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

ROOT = Path.cwd()
NOTES_FILE = ROOT / 'data' / 'followup_notes.csv'
APPOINTMENTS_FILE = ROOT / 'appointments_clean.csv'
MODEL_NAME = 'gemini-3.5-flash-lite'  # Lower-latency current Flash-Lite fallback for note classification.
API_KEY = input('Paste your Gemini API key (runtime only): ')
client = genai.Client(
    api_key=API_KEY,
    http_options=types.HttpOptions(
        timeout=60000,  # 60-second request timeout (milliseconds)
        retry_options=types.HttpRetryOptions(attempts=1)  # avoid long automatic retry waits
    )
)
API_CALLS = 0
ALLOWED_ROOT_CAUSES = [
    'forgot_or_no_reminder', 'distance_or_transport', 'cost_or_fee',
    'illness_or_emergency', 'work_or_schedule', 'other'
]

SCHEMA = {
    'type': 'object',
    'properties': {
        'root_cause': {'type': 'string'},
        'confidence': {'type': 'number'},
        'summary': {'type': 'string'},
        'recommended_action': {'type': 'string'}
    },
    'required': ['root_cause', 'confidence', 'summary', 'recommended_action']
}

SYSTEM_TEXT = (
    'You are a careful outpatient-clinic call-centre note analyst. '
    'Classify only what the note supports. Do not infer medical details or invent facts. '
    'Use other when the cause is ambiguous or does not fit a listed category.'
)


In [ ]:
MODEL_NAME = "gemini-3.1-flash-lite"
print("Current model:", MODEL_NAME)

In [ ]:
MODEL_NAME = "gemini-3.6-flash"

print("Selected model:", MODEL_NAME)

## Task 4.1 — One validated call: N001

The CRAFT prompt supplies the context, role, action, tone and required output format. The schema keeps `root_cause` as a plain string; the allowed-value rule is enforced afterward by Pydantic/application validation.

In [ ]:
def build_craft_prompt(note_text):
    return f"""Context: CareLoop Clinics is reviewing short call-centre notes recorded after a patient missed an outpatient appointment.
Role: You are a careful clinical-administration data analyst, not a clinician.
Action: Classify the main reason supported by the note. Choose exactly one root_cause from this list: {', '.join(ALLOWED_ROOT_CAUSES)}. Use 'other' rather than guessing when the note is unclear or does not support a listed reason. Keep summary to at most 25 words. Recommend a practical administrative follow-up, not medical advice.
Tone: Neutral, concise, factual, non-judgmental.
Format: Return only the required JSON object with exactly these fields: root_cause (string), confidence (number from 0 to 1), summary (string, at most 25 words), recommended_action (string).

Note to classify:
{note_text}"""


def generate_json(prompt):
    """One Gemini Flash-Lite request using the required JSON schema and no thinking config."""
    global API_CALLS
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_TEXT,
        temperature=0,
        max_output_tokens=500,
        response_mime_type='application/json',
        response_schema=SCHEMA
    )
    API_CALLS += 1
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=config
    )
    return response.text or ''


def analyse_note(note_text):
    raw_text = generate_json(build_craft_prompt(note_text))
    return json.loads(raw_text)

n001 = pd.read_csv(NOTES_FILE).loc[lambda d: d['note_id'].eq('N001'), 'note_text'].iloc[0]
parsed_n001 = analyse_note(n001)
print(parsed_n001)
assert set(parsed_n001) == {'root_cause', 'confidence', 'summary', 'recommended_action'}


In [ ]:
note = notes_df.loc[
    notes_df["note_id"] == "N005", "note_text"
].iloc[0]

try:
    raw = generate_json(build_craft_prompt(note))
    result = json.loads(raw)
    validated = FollowupInsight(**result)
    validate_allowed_root_cause(validated)

    print("SUCCESS:", result)

except Exception as e:
    print("FAILED:", type(e).__name__, str(e)[:500])

In [ ]:
MODEL_NAME = "gemini-3.5-flash-lite"

client = genai.Client(
    api_key=API_KEY,
    http_options=types.HttpOptions(
        timeout=30000,
        retry_options=types.HttpRetryOptions(attempts=1)
    )
)

print("Active model:", MODEL_NAME)

## Task 4.2 — Pydantic validation and process first 10 notes

**Reliability fallback:** if the default Flash endpoint returns repeated 503/504 errors, this notebook uses `gemini-3.5-flash-lite` with JSON mode and no thinking configuration. This is a current low-latency Flash-Lite model with a listed free tier. The Task 4.2 cell is resumable and writes a partial CSV after each note. Any request that fails twice is explicitly printed in `api_errors`; it is not presented as a model-generated reason.


In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)


def validate_allowed_root_cause(data):
    if data.root_cause not in ALLOWED_ROOT_CAUSES:
        raise ValueError(f'root_cause must be one of {ALLOWED_ROOT_CAUSES}; received {data.root_cause!r}')
    return data

notes_df = pd.read_csv(NOTES_FILE).head(10)
partial_path = ROOT / 'notes_analysed_partial.csv'
output_path = ROOT / 'notes_analysed.csv'
columns = [
    'note_id', 'appointment_id', 'root_cause', 'confidence', 'summary',
    'recommended_action', 'validation_status', 'raw_response'
]

# Resume safely: prefer existing partial output; otherwise reuse in-memory records.
if partial_path.exists():
    prior_df = pd.read_csv(partial_path, keep_default_na=False)
    analysed_records = prior_df.to_dict('records')
elif 'analysed_records' not in globals():
    analysed_records = []

records_by_id = {r['note_id']: r for r in analysed_records}
calls_before_task_42 = API_CALLS
api_errors = {}

for _, row in notes_df.iterrows():
    note_id = row['note_id']
    note_text = row['note_text']
    previous = records_by_id.get(note_id)

    # Preserve a genuine response (valid or invalid) and the required blank-note row.
    if previous is not None:
        if previous.get('validation_status') == 'skipped_blank':
            continue
        if str(previous.get('raw_response', '')).strip():
            continue

    base = {
        'note_id': note_id,
        'appointment_id': row['appointment_id'],
        'root_cause': '', 'confidence': '', 'summary': '',
        'recommended_action': '', 'validation_status': '', 'raw_response': ''
    }

    if pd.isna(note_text) or str(note_text).strip() == '':
        base.update({
            'root_cause': 'other', 'confidence': 0.0,
            'summary': 'No note recorded',
            'recommended_action': 'Re-call patient',
            'validation_status': 'skipped_blank', 'raw_response': ''
        })
        records_by_id[note_id] = base
        analysed_records = list(records_by_id.values())
        pd.DataFrame(analysed_records, columns=columns).sort_values('note_id').to_csv(partial_path, index=False)
        print(note_id, 'blank note: skipped API call', flush=True)
        continue

    raw_text = ''
    completed = False
    for attempt in range(2):
        try:
            print(f'Gemini Flash-Lite: {note_id}, attempt {attempt + 1}', flush=True)
            raw_text = generate_json(build_craft_prompt(str(note_text)))
            response_data = json.loads(raw_text)
            if not isinstance(response_data, dict):
                raise ValueError('Model response JSON must be an object')

            for key in ['root_cause', 'confidence', 'summary', 'recommended_action']:
                if key in response_data:
                    base[key] = response_data[key]

            validated = FollowupInsight(**response_data)
            validate_allowed_root_cause(validated)
            base['validation_status'] = 'valid'
            completed = True
            break
        except Exception as exc:
            raw_text = raw_text or ''
            api_errors[note_id] = str(exc)
            print(f'{note_id} attempt {attempt + 1} failed: {exc}', flush=True)
            if attempt == 0:
                time.sleep(8)

    # If Gemini returned JSON but it failed validation, retain the raw reply as invalid.
    # If the API itself failed twice, mark invalid with blank raw_response and disclose it in api_errors.
    if not completed:
        base['validation_status'] = 'invalid'
    base['raw_response'] = raw_text
    records_by_id[note_id] = base
    analysed_records = list(records_by_id.values())
    pd.DataFrame(analysed_records, columns=columns).sort_values('note_id').to_csv(partial_path, index=False)
    print(f'Saved {len(records_by_id)}/10 rows; {note_id}: {base["validation_status"]}', flush=True)
    time.sleep(13)

notes_analysed = pd.DataFrame(
    [records_by_id[n['note_id']] for n in notes_df.to_dict('records')],
    columns=columns
)
notes_analysed.to_csv(output_path, index=False)

print('\nTask 4.2 processing finished.')
print('Rows:', len(notes_analysed))
print('New API calls in this run:', API_CALLS - calls_before_task_42)
print('Rows with API errors (retry again if time allows):', sorted(api_errors))
print(notes_analysed[['note_id', 'root_cause', 'confidence', 'validation_status']].to_string(index=False))


## Task 4.3 — Deliberately invalid replies and prompt comparison

In [ ]:
bad_replies = [
    {
        'root_cause': 'cost_or_fee', 'confidence': 1.7,
        'summary': 'Fee was too high', 'recommended_action': 'Offer fee information'
    },
    {
        'root_cause': 'work_or_schedule', 'confidence': 0.8,
        'summary': 'Work conflict was reported'
        # recommended_action intentionally missing
    }
]
for number, bad_reply in enumerate(bad_replies, start=1):
    try:
        FollowupInsight(**bad_reply)
    except ValidationError as exc:
        print(f'Bad reply {number} validation error:\n{exc}\n')

# V1: intentionally minimal prompt. V2: reuse the CRAFT replies already generated in Task 4.2.
def get_raw_root_cause(raw_text):
    try:
        parsed = json.loads(raw_text)
        value = parsed.get('root_cause', '') if isinstance(parsed, dict) else ''
        return value, value in ALLOWED_ROOT_CAUSES
    except Exception:
        return '', False

ab_rows = []
for note_id in ['N007', 'N009']:
    note_text = notes_df.loc[notes_df['note_id'].eq(note_id), 'note_text'].iloc[0]
    v1_prompt = 'Classify this note. Return JSON matching the required schema.\n\n' + str(note_text)
    print(f'Sending V1 Gemini request for {note_id}...', flush=True)
    v1_raw = generate_json(v1_prompt)
    print(f'Finished V1 Gemini request for {note_id}.', flush=True)
    v1_cause, v1_allowed = get_raw_root_cause(v1_raw)

    v2_row = notes_analysed.loc[notes_analysed['note_id'].eq(note_id)].iloc[0]
    v2_cause, v2_allowed = get_raw_root_cause(v2_row['raw_response'])
    ab_rows.append({'note_id': note_id, 'prompt_version': 'V1 basic', 'root_cause': v1_cause, 'allowed_value': v1_allowed})
    ab_rows.append({'note_id': note_id, 'prompt_version': 'V2 CRAFT (reused Task 4.2)', 'root_cause': v2_cause, 'allowed_value': v2_allowed})

prompt_comparison = pd.DataFrame(ab_rows)
print(prompt_comparison.to_string(index=False))
print('Deploy V2 CRAFT: it specifies the allowed categories and requires other rather than guessing when the note is ambiguous.')
print('Total API calls across Tasks 4.1–4.3:', API_CALLS)


## Task 4.4 — Merge notes with appointment data and write recommendation evidence

In [ ]:
appointments_clean = pd.read_csv(APPOINTMENTS_FILE)
merged_notes = notes_analysed.merge(
    appointments_clean[['appointment_id', 'reminder_sent', 'no_show', 'clinic_id', 'lead_days']],
    on='appointment_id', how='left', validate='one_to_one'
)
print('Merged note rows:', len(merged_notes))
print('Root cause by reminder status:\n', pd.crosstab(merged_notes['root_cause'], merged_notes['reminder_sent']))
print('Notes linked to reminder_sent=No:', int((merged_notes['reminder_sent'] == 'No').sum()), 'of', len(merged_notes))
print('Final note output file:', ROOT / 'notes_analysed.csv')


In [ ]:
from pathlib import Path
import pandas as pd

output = Path(ROOT) / "notes_analysed.csv"

print("PART 4 VERIFICATION")
print("-------------------")
print("Notes CSV exists:", output.exists())
print("Task 4.3 completed:", "prompt_comparison" in globals())
print("Task 4.4 completed:", "merged_notes" in globals())

if output.exists():
    result = pd.read_csv(output, keep_default_na=False)

    print("\nCSV shape:", result.shape)
    print("\nNote classifications:")
    print(result[
        ["note_id", "root_cause", "validation_status"]
    ].to_string(index=False))

    print("\nValidation counts:")
    print(result["validation_status"].value_counts().to_dict())

    print("\nN004 skipped correctly:",
          result.loc[
              result["note_id"] == "N004",
              "validation_status"
          ].tolist())

In [ ]:
print("TASK 4.3 — PROMPT COMPARISON")
print(prompt_comparison.to_string(index=False))

print("\nTOTAL GEMINI API CALLS:", API_CALLS)

print("\nTASK 4.4 — REMINDER CROSS-TABULATION")
print(pd.crosstab(
    merged_notes["root_cause"],
    merged_notes["reminder_sent"]
))

print("\nINVALID RESPONSE CHECK")
print(
    notes_analysed.loc[
        notes_analysed["validation_status"] == "invalid",
        ["note_id", "raw_response"]
    ].to_string(index=False)
)

In [ ]:
from google.colab import files
files.download("notes_analysed.csv")

In [ ]:
print("Total API calls:", API_CALLS)
print(prompt_comparison.to_string(index=False))
print("Merged rows:", len(merged_notes))

print(pd.crosstab(
    merged_notes["root_cause"],
    merged_notes["reminder_sent"]
))

## Submission and privacy check

The final notebook must not contain the API key. The key is entered via `input()` at runtime. Clear outputs before committing if any output could contain the key. Commit the generated `notes_analysed.csv` after the run and copy the measured Part 4 counts and crosstab into `README.md`.